# Example 13 - Evaluation within Industry Groups

Rank and settle a factor within industry groups.


In [ ]:
import asyncio
import datetime as dt
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.portfolio.rankbuilder import rank_build
from alphamind.settlement.simplesettle import simple_settle

labels = panel["industry"].to_pylist()
group_labels = [
    f"{trade_date.isoformat()}:{label}"
    for trade_date, label in zip(panel["trade_date"].to_pylist(), labels)
]
groups = np.unique(group_labels, return_inverse=True)[1]
signal = panel["factor"].to_numpy(zero_copy_only=False)
selected = rank_build(signal, use_rank=2, groups=groups).flatten().astype(float)
weights = selected / 6.0
settlement = await asyncio.to_thread(
    simple_settle,
    weights,
    panel["dx"].to_numpy(zero_copy_only=False),
    groups=labels,
)
assert selected.sum() == n_dates * 6
settlement
